In [4]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import StratifiedKFold

from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import wilcoxon

In [6]:
# LOAD DATASET AND DEFINE DATASET
df = pd.read_csv("INCART 2-lead Arrhythmia Database.csv")
print(df.shape)
df.head()

(175729, 34)


,record,type,0_pre-RR,0_post-RR,0_pPeak,0_tPeak,0_rPeak,0_sPeak,0_qPeak,0_qrs_interval,...,1_qPeak,1_qrs_interval,1_pq_interval,1_qt_interval,1_st_interval,1_qrs_morph0,1_qrs_morph1,1_qrs_morph2,1_qrs_morph3,1_qrs_morph4
0,I01,N,163,165,0.069610,-0.083281,0.614133,-0.392761,0.047159,15,...,-0.023370,14,3,23,6,-0.023370,-0.011650,0.082608,0.101373,-0.183387
1,I01,N,165,166,-0.097030,0.597254,-0.078704,-0.078704,-0.137781,3,...,0.081637,15,5,27,7,0.081637,0.102992,0.191225,0.217544,-0.068248
2,I01,N,166,102,0.109399,0.680528,-0.010649,-0.010649,-0.720620,6,...,-0.148539,33,13,52,6,-0.148539,-0.060620,0.081080,0.204400,0.335172
3,I01,VEB,102,231,0.176376,0.256431,-0.101098,-0.707525,-0.101098,4,...,0.046898,21,9,34,4,0.046898,0.083728,0.279512,0.526785,0.450969
4,I01,N,231,165,0.585577,0.607461,-0.083499,-0.083499,-0.167858,3,...,-0.112552,32,5,43,6,-0.112552,0.012989,0.091491,0.134004,0.265232


In [7]:
#DATA PREPROCESSING
# Remove duplicate rows
df = df.drop_duplicates()

# Missing values
df.isnull().sum()

# Remove identifier column
df.drop(columns=['record'], inplace=True)

# Separate X and y
X = df.drop("type", axis=1)
y = df["type"]

# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [8]:
models = {

'KNN':

KNeighborsClassifier(
    n_neighbors=3
),

'MLP':

MLPClassifier(
    hidden_layer_sizes=(50,),
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42
)
}

In [9]:
cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)


In [10]:
kappa=make_scorer(cohen_kappa_score)
mcc=make_scorer(matthews_corrcoef)
scoring={
    'accuracy':'accuracy',
    'precision':'precision',
    'recall':'recall',
    'f1':'f1',
    'roc_auc':'roc_auc',
    'kappa':kappa,
    'mcc':mcc
}

In [14]:
for metric in metrics:

    metrics[metric]['KNN'] = []

    metrics[metric]['MLP'] = []


for train_idx, test_idx in cv.split(X,y):

    X_train = X.iloc[train_idx]

    X_test = X.iloc[test_idx]

    y_train = y[train_idx]

    y_test = y[test_idx]


    for name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_test
        )

        prob = model.predict_proba(
            X_test
        )


        acc = accuracy_score(
            y_test,
            pred
        )

        prec = precision_score(
            y_test,
            pred,
            average='weighted',
            zero_division=0
        )

        rec = recall_score(
            y_test,
            pred,
            average='weighted'
        )

        f1 = f1_score(
            y_test,
            pred,
            average='weighted'
        )

        roc = roc_auc_score(
            y_test,
            prob,
            multi_class='ovr',
            average='weighted'
        )

        kap = cohen_kappa_score(
            y_test,
            pred
        )

        mcc = matthews_corrcoef(
            y_test,
            pred
        )

        cm = confusion_matrix(
            y_test,
            pred
        )

        sensitivity = np.diag(cm) / cm.sum(axis=1)

        gmean = np.prod(
            sensitivity
        )**(
            1/len(sensitivity)
        )


        metrics['Accuracy'][name].append(acc)

        metrics['Precision'][name].append(prec)

        metrics['Recall'][name].append(rec)

        metrics['F1'][name].append(f1)

        metrics['ROC'][name].append(roc)

        metrics['Kappa'][name].append(kap)

        metrics['MCC'][name].append(mcc)

        metrics['GMean'][name].append(gmean)



In [16]:
# =====================
# Wilcoxon
# =====================

results = []

for metric in metrics:

    stat, p = wilcoxon(

        metrics[metric]['KNN'],

        metrics[metric]['MLP']

    )

    knn_mean = np.mean(

        metrics[metric]['KNN']

    )

    mlp_mean = np.mean(

        metrics[metric]['MLP']

    )

    results.append({

        'Metric': metric,

        'KNN_Mean': round(knn_mean,4),

        'MLP_Mean': round(mlp_mean,4),

        'Statistic': round(stat,4),

        'P_Value': round(p,6),

        'Significant': p < 0.05

    })


result_df = pd.DataFrame(

    results

)


print()

print(result_df)


result_df.to_excel(

    'WilcoxonResults.xlsx',

    index=False

)


      Metric  KNN_Mean  MLP_Mean  Statistic  P_Value  Significant
0   Accuracy    0.9885    0.9939        0.0      0.5        False
1  Precision    0.9881    0.9935        0.0      0.5        False
2     Recall    0.9885    0.9939        0.0      0.5        False
3         F1    0.9883    0.9935        0.0      0.5        False
4        ROC    0.9848    0.9983        0.0      0.5        False
5      Kappa    0.9478    0.9726        0.0      0.5        False
6        MCC    0.9480    0.9726        0.0      0.5        False
7      GMean    0.0000    0.2644        0.0      1.0        False
